# Normalizing flows

_Modern Deep Learning & AI_

**Bend a plain Gaussian through an invertible map into any shape, and read off the exact probability.**

A normalizing flow builds a complex probability distribution out of a simple one.

     Start with an easy density, a standard Gaussian. Pass each sample through an invertible transform $g$. The Gaussian gets stretched and folded into a rich, multi-peaked shape.

---

This notebook builds a normalizing flow one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Define an invertible coupling layer

A coupling layer is the building block of a flow. It splits the input in half: the first half `xa` passes through untouched, while a small network reads `xa` and predicts an affine transform (a scale `log_s` and a shift `t`) to apply to the second half `xb`. Because `xa` is left alone, the transform is exactly invertible, and the log-determinant of its Jacobian is simply the sum of `log_s`.

In [ ]:
import torch
import torch.nn as nn

class CouplingLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.half = dim // 2
        # net reads the untouched half, outputs scale (log s) and shift (t)
        self.net = nn.Sequential(
            nn.Linear(self.half, 16),
            nn.ReLU(),
            nn.Linear(16, 2 * (dim - self.half)),
        )

    def forward(self, x):
        xa = x[:, :self.half]        # untouched half
        xb = x[:, self.half:]        # transformed half

        log_s, t = self.net(xa).chunk(2, dim=1)
        log_s = torch.tanh(log_s)                  # stabilize the scale

        yb = xb * torch.exp(log_s) + t             # affine transform on xb
        y = torch.cat([xa, yb], dim=1)

        log_det = log_s.sum(dim=1)                 # Jacobian log-det
        return y, log_det

### Step 2 — Push base samples through the layer

We draw a batch of base samples from a standard Gaussian `N(0, I)` and pass them through the coupling layer. The output `y` is the transformed point, and `log_det` records how much the layer locally stretched or shrank space — we'll need that to get the probability exactly right.

In [ ]:
torch.manual_seed(0)
dim = 4

layer = CouplingLayer(dim)

x = torch.randn(8, dim)            # base samples ~ N(0, I)
y, log_det = layer(x)

print("y shape:", y.shape, "log_det[0]:", round(log_det[0].item(), 4))

### Step 3 — Compute the exact log-likelihood

This is the payoff of an invertible map: the change-of-variables formula gives the *exact* density. The log-likelihood of a transformed point is the base Gaussian log-probability of `y` plus the log-determinant from Step 2. No approximation — flows give you a tractable likelihood.

In [ ]:
# base log-prob of y under a standard Gaussian, summed over dimensions
base_logp = (-0.5 * y.pow(2) - 0.5 * torch.log(torch.tensor(2 * 3.14159))).sum(1)

# exact log-likelihood under the flow = base log-prob + log|det Jacobian|
log_prob = base_logp + log_det

print("log_prob[0]:", round(log_prob[0].item(), 4))

## Visualize the data & results

_Can an invertible map turn one Gaussian hump into the real bimodal spread of digits 0 and 1?_

### Step 1 — Build a real bimodal target from the digits

To get a genuinely two-peaked target, we run PCA on the digit images and keep the first component's scores for just digits 0 and 1. Because those two digits look very different, their projected scores form two separated clusters. We standardize the scores and bin them into a histogram density.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# REAL bimodal target: PCA component-1 scores of digits 0 and 1
digits = load_digits()
Z = PCA(n_components=2).fit_transform(digits.data / 16.0)

mask = (digits.target == 0) | (digits.target == 1)
v = Z[mask, 0]
v = (v - v.mean()) / v.std()        # standardize

hist, edges = np.histogram(v, bins=18, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])

### Step 2 — Push a Gaussian through an invertible map

Here is a flow in one line: take a base Gaussian over `u` and apply the invertible transform `x = u + sep * tanh(u)`. The change-of-variables formula says the new density is the base density divided by `|g'(u)|`, the absolute derivative of the map. The `tanh` term pulls probability mass away from the center, splitting the single hump into two peaks.

In [ ]:
sep = 1.5
u = np.linspace(-3.5, 3.5, 41)

pu = np.exp(-0.5 * u ** 2) / np.sqrt(2 * np.pi)     # base Gaussian density

x = u + sep * np.tanh(u)                            # invertible transform g(u)
g_prime = 1.0 + sep * (1.0 - np.tanh(u) ** 2)       # derivative g'(u)

px = pu / np.abs(g_prime)                           # transformed density

### Step 3 — Compare the flow density to the real data

We overlay three curves: the base Gaussian, the flow's transformed density, and the real digit-0-vs-1 histogram. The flow's two peaks should line up with the two clusters in the data — a one-parameter invertible map reshaping a plain Gaussian into the real bimodal spread.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(u, pu, color="#4ea1ff", label="base p_u(u): Gaussian")
ax.plot(x, px, color="#7ee787", label="flow p_x(x): two peaks")
ax.plot(centers, hist, color="#ffb454", label="real digit 0 and 1 scores")

ax.set_xlabel("value")
ax.set_ylabel("probability density")
ax.set_title("flow matches the real bimodal target")
ax.legend()

plt.show()